In [36]:
import os
import mysql.connector
from dotenv import load_dotenv

load_dotenv()

def get_connection():
    return mysql.connector.connect(
        host=os.getenv("DB_HOST"),
        port=int(os.getenv("DB_PORT")),
        database=os.getenv("DB_NAME"),
        user=os.getenv("DB_USER"),
        password=os.getenv("DB_PASSWORD"),
        ssl_ca=os.getenv("DB_SSL_CA")
    )

# Prueba
conn = get_connection()
print("✅ Conexión exitosa")
conn.close()

✅ Conexión exitosa


In [37]:
import pandas as pd
import google.genai as genai
from tabulate import tabulate
from IPython.display import display

print("✅ Librerías cargadas")

✅ Librerías cargadas


In [38]:
from google import genai

client = genai.Client(api_key=os.getenv("LLM_API_KEY"))

print("✅ Gemini configurado")

✅ Gemini configurado


In [52]:
SYSTEM_PROMPT = """
Sos un experto en bases de datos MySQL. Tu única tarea es convertir preguntas 
en español a consultas SQL válidas para la base de datos 'biblioia'.

REGLAS:
- Respondé ÚNICAMENTE con la consulta SQL, sin explicaciones ni comentarios.
- No uses bloques de código ni backticks.
- Si la pregunta no se puede responder con el esquema dado, respondé: 
  SELECT 'No puedo responder esa pregunta con los datos disponibles';

=== ESQUEMA ===

GENERO (id_genero INT PK, nombre VARCHAR(60) UNIQUE NOT NULL, descripcion VARCHAR(255))

AUTOR (id_autor INT PK, nombre VARCHAR(80) NOT NULL, apellido VARCHAR(80) NOT NULL, nacionalidad VARCHAR(60))

LIBRO (isbn VARCHAR(20) PK, titulo VARCHAR(200) NOT NULL, anio_publicacion YEAR,
       stock_total SMALLINT, stock_disponible SMALLINT)
  -- stock_disponible <= stock_total siempre
  -- stock_disponible >= 0 siempre

LIBRO_AUTOR (isbn FK->LIBRO, id_autor FK->AUTOR) -- relación N:M entre LIBRO y AUTOR

LIBRO_GENERO (isbn FK->LIBRO, id_genero FK->GENERO) -- relación N:M entre LIBRO y GENERO

SOCIO (id_socio INT PK, dni VARCHAR(15) UNIQUE, nombre VARCHAR(80), apellido VARCHAR(80),
       email VARCHAR(120) UNIQUE, fecha_alta DATE, estado VARCHAR(12))
  -- estado puede ser: 'ACTIVO', 'SUSPENDIDO', 'BAJA'

EJEMPLAR (id_ejemplar INT PK, isbn FK->LIBRO, nro_ejemplar SMALLINT, estado_fisico VARCHAR(12))
  -- estado_fisico puede ser: 'BUENO', 'DETERIORADO', 'BAJA'

PRESTAMO (id_prestamo INT PK, id_socio FK->SOCIO, id_ejemplar FK->EJEMPLAR,
          fecha_prestamo DATE, fecha_vencimiento DATE, fecha_devolucion DATE NULL, estado VARCHAR(12))
  -- estado puede ser: 'ACTIVO', 'DEVUELTO', 'VENCIDO'
  -- fecha_devolucion es NULL mientras el préstamo está activo

SANCION (id_sancion INT PK, id_socio FK->SOCIO, tipo VARCHAR(20), fecha_inicio DATE, fecha_fin DATE, motivo VARCHAR(255))
  -- tipo puede ser: 'MORA', 'DAÑO', 'PERDIDA', 'OTRO'
  -- una sanción está activa cuando fecha_fin >= CURRENT_DATE

AUDITORIA_PRESTAMOS (id_audit INT PK, id_prestamo INT, operacion VARCHAR(10),
                     estado_nuevo VARCHAR(12), estado_viejo VARCHAR(12), usuario_bd VARCHAR(80), fecha_hora DATETIME)
=== EJEMPLOS ===

Pregunta: ¿Cuáles son los 5 libros más prestados este año?
SQL: SELECT L.isbn, L.titulo, COUNT(P.id_prestamo) AS cantidad_prestamos FROM LIBRO L JOIN EJEMPLAR E ON L.isbn = E.isbn JOIN PRESTAMO P ON E.id_ejemplar = P.id_ejemplar WHERE YEAR(P.fecha_prestamo) = YEAR(CURRENT_DATE) GROUP BY L.isbn, L.titulo ORDER BY cantidad_prestamos DESC LIMIT 5;

Pregunta: ¿Qué socios tienen préstamos vencidos en este momento?
SQL: SELECT DISTINCT S.id_socio, S.dni, S.nombre, S.apellido FROM SOCIO S JOIN PRESTAMO P ON S.id_socio = P.id_socio WHERE P.estado = 'ACTIVO' AND P.fecha_vencimiento < CURRENT_DATE;

Pregunta: ¿Qué libros de ciencia ficción están disponibles para prestar?
SQL: SELECT L.isbn, L.titulo, L.stock_disponible FROM LIBRO L JOIN LIBRO_GENERO LG ON L.isbn = LG.isbn JOIN GENERO G ON LG.id_genero = G.id_genero WHERE G.nombre LIKE '%ciencia ficción%' AND L.stock_disponible > 0;
"""


In [ ]:
def text_to_sql(pregunta: str) -> str:
    respuesta = client.models.generate_content(
       model="gemini-2.5-flash",
        contents=SYSTEM_PROMPT + f"\nPregunta: {pregunta}\nSQL:"
    )
    sql = respuesta.text.strip()
    # Por las dudas, limpiamos backticks que Gemini a veces agrega
    sql = sql.replace("```sql", "").replace("```", "").strip()
    return sql


SELECT COUNT(id_socio) FROM SOCIO;


In [39]:
def ejecutar_consulta(sql: str) -> pd.DataFrame:
    conn = get_connection()
    try:
        df = pd.read_sql(sql, conn)
        return df
    except Exception as e:
        return pd.DataFrame({"Error": [str(e)]})
    # cirra la conexion 
    finally:
        conn.close()

In [50]:
def agente_responder(pregunta: str, mostrar_sql: bool = True):
    print(f"\n Pregunta: {pregunta}")
    print("-" * 60)
    
    sql = text_to_sql(pregunta)
    
    if mostrar_sql:
        print(f" SQL generado:\n{sql}")
        print("-" * 60)
    
    df = ejecutar_consulta(sql)
    
    if df.empty:
        print(" La consulta no devolvió resultados.")
    else:
        print(f" Resultado ({len(df)} filas):")
        display(df)
    
    return df

In [51]:
agente_responder("¿Cuantos socios hay en total?")


 Pregunta: ¿Cuantos socios hay en total?
------------------------------------------------------------
 SQL generado:
SELECT COUNT(*) FROM SOCIO;
------------------------------------------------------------


C:\Users\miran\AppData\Local\Temp\ipykernel_21420\3436965000.py:4: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn)


 Resultado (1 filas):


,COUNT(*)
0,0


,COUNT(*)
0,0
